# Face Mask and Emotion Detection - Model Training and Evaluation

in this notebook i will:
- build a simple baseline CNN model
- build a better model using transfer learning (MobileNetV2)
- fine-tune the transfer learning model
- handle overfitting with dropout, early stopping, and data augmentation
- evaluate both models and compare them

note: run notebook 01 first to download the datasets,
or just run the download cells below again.

## install and import libraries

In [ ]:
!pip install tensorflow opencv-python-headless scikit-learn albumentations opendatasets mtcnn -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve,
                              accuracy_score, f1_score,
                              precision_score, recall_score)
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

print('tensorflow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## download datasets (skip if already downloaded from notebook 1)

In [ ]:
import opendatasets as od

MASK_IMG_DIR  = Path('face-mask-detection/images')
MASK_ANNO_DIR = Path('face-mask-detection/annotations')
FER_TRAIN_DIR = Path('fer2013/train')
FER_TEST_DIR  = Path('fer2013/test')

EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
IMG_SIZE   = (224, 224)   # MobileNetV2 needs 224x224
BATCH_SIZE = 32

os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('paths set up')

## prepare the mask dataset for classification

the mask dataset has XML files with bounding boxes.
i need to:
1. read each XML
2. crop the face out of the image
3. save it to a folder by class (with_mask / without_mask)

then keras ImageDataGenerator can just read from those folders.

In [ ]:
def crop_faces_from_annotations(img_dir, anno_dir, output_dir):
    # label mapping - merge 'mask_weared_incorrect' into 'without_mask'
    label_map = {
        'with_mask': 'with_mask',
        'without_mask': 'without_mask',
        'mask_weared_incorrect': 'without_mask'
    }

    # create output folders
    for cls in ['with_mask', 'without_mask']:
        Path(output_dir, cls).mkdir(parents=True, exist_ok=True)

    saved = 0
    for xml_file in sorted(Path(anno_dir).glob('*.xml')):
        tree = ET.parse(xml_file)
        root = tree.getroot()

        filename = root.find('filename').text
        img_path = Path(img_dir) / filename

        if not img_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        for i, obj in enumerate(root.findall('object')):
            label_str = obj.find('name').text.strip().lower()
            cls = label_map.get(label_str)
            if cls is None:
                continue

            # get bounding box
            box = obj.find('bndbox')
            x1 = int(float(box.find('xmin').text))
            y1 = int(float(box.find('ymin').text))
            x2 = int(float(box.find('xmax').text))
            y2 = int(float(box.find('ymax').text))

            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            # save to output folder
            out_path = Path(output_dir) / cls / f'{xml_file.stem}_{i}.jpg'
            cv2.imwrite(str(out_path), crop)
            saved += 1

    print(f'saved {saved} face crops to {output_dir}')
    return saved

MASK_FACES_DIR = Path('data/processed/mask_faces')
crop_faces_from_annotations(MASK_IMG_DIR, MASK_ANNO_DIR, MASK_FACES_DIR)

## create data generators

i use ImageDataGenerator from keras to:
- load images in batches
- apply augmentation during training (flip, rotation, brightness)
- split 20% for validation

In [ ]:
# generator for mask dataset
# training set: with augmentation
# validation set: just rescale

mask_train_gen_obj = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    validation_split=0.2
)

mask_val_gen_obj = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

mask_train = mask_train_gen_obj.flow_from_directory(
    MASK_FACES_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=42
)

mask_val = mask_val_gen_obj.flow_from_directory(
    MASK_FACES_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=42
)

print('mask class mapping:', mask_train.class_indices)
print('training batches:', len(mask_train))
print('validation batches:', len(mask_val))

In [ ]:
# generator for emotion dataset
# fer2013 already has train/test split so i use them directly

emo_train_gen_obj = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1
)

emo_val_gen_obj = ImageDataGenerator(rescale=1./255)

emo_train = emo_train_gen_obj.flow_from_directory(
    FER_TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

emo_val = emo_val_gen_obj.flow_from_directory(
    FER_TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

print('emotion class mapping:', emo_train.class_indices)

# compute class weights for the emotion dataset (it's imbalanced)
all_emo_labels = emo_train.labels
emo_weights = compute_class_weight('balanced', classes=np.unique(all_emo_labels), y=all_emo_labels)
emo_class_weight = dict(enumerate(emo_weights))
print('class weights:', emo_class_weight)

## Step 7 - Build and Train Models

### Part A: Baseline CNN (simple model for comparison)

first i build a simple CNN from scratch.
this gives us a baseline accuracy to compare against.

In [ ]:
# simple CNN baseline - 3 conv blocks + dense layers
def make_baseline_cnn(num_classes=1, binary=True):
    if binary:
        output_units = 1
        output_activation = 'sigmoid'
        loss = 'binary_crossentropy'
    else:
        output_units = num_classes
        output_activation = 'softmax'
        loss = 'categorical_crossentropy'

    model = models.Sequential([
        # block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
        layers.MaxPooling2D(2, 2),

        # block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        # block 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        # fully connected
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),  # dropout to prevent overfitting
        layers.Dense(output_units, activation=output_activation)
    ])

    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
    return model

baseline_mask  = make_baseline_cnn(binary=True)
baseline_emo   = make_baseline_cnn(num_classes=7, binary=False)

baseline_mask.summary()

In [ ]:
# train baseline mask model
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print('training baseline mask model...')
history_baseline_mask = baseline_mask.fit(
    mask_train,
    validation_data=mask_val,
    epochs=15,
    callbacks=[early_stop],
    verbose=1
)
print('done')

In [ ]:
# train baseline emotion model
print('training baseline emotion model...')
history_baseline_emo = baseline_emo.fit(
    emo_train,
    validation_data=emo_val,
    epochs=15,
    class_weight=emo_class_weight,
    callbacks=[early_stop],
    verbose=1
)
print('done')

### Part B: Transfer Learning with MobileNetV2

instead of training from scratch, i use MobileNetV2 which was already trained on ImageNet (1.4 million images).

my approach:
1. freeze all MobileNetV2 layers and add a new head on top
2. train only the new head for 10 epochs
3. unfreeze the last 30 layers and fine-tune with a small learning rate

In [ ]:
def make_mobilenet_model(num_classes=1, binary=True, dropout=0.4):
    # load MobileNetV2 without the top classification layer
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False  # freeze it for now

    if binary:
        output_units = 1
        output_act   = 'sigmoid'
        loss         = 'binary_crossentropy'
    else:
        output_units = num_classes
        output_act   = 'softmax'
        loss         = 'categorical_crossentropy'

    # build custom head on top of MobileNetV2
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)  # L2 regularization
    x = layers.Dropout(dropout)(x)  # dropout
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(output_units, activation=output_act)(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=loss, metrics=['accuracy'])
    return model, base

mask_model, mask_base  = make_mobilenet_model(binary=True)
emo_model, emo_base    = make_mobilenet_model(num_classes=7, binary=False)

print('total parameters in mask model:', mask_model.count_params())
mask_model.summary()

In [ ]:
# callbacks for training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),  # reduce LR if stuck
    ModelCheckpoint('models/mask_model_best.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print('callbacks ready')

In [ ]:
# phase 1: train only the new head (base is frozen)
print('=== phase 1: training new head (base frozen) for mask model ===')
history_mask_p1 = mask_model.fit(
    mask_train,
    validation_data=mask_val,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# phase 2: unfreeze last 30 layers and fine-tune
print('=== phase 2: fine-tuning (last 30 layers unfrozen) ===')
mask_base.trainable = True

# keep early layers frozen (they hold general features)
for layer in mask_base.layers[:-30]:
    layer.trainable = False

# use a much smaller learning rate for fine-tuning
mask_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                   loss='binary_crossentropy', metrics=['accuracy'])

history_mask_p2 = mask_model.fit(
    mask_train,
    validation_data=mask_val,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

# combine both phase histories
def merge_hist(h1, h2):
    result = {}
    for k in h1.history:
        result[k] = h1.history[k] + h2.history[k]
    return result

history_mask = merge_hist(history_mask_p1, history_mask_p2)
print('mask model training finished')

In [ ]:
# update callbacks for emotion model
callbacks_emo = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('models/emotion_model_best.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

# phase 1 for emotion
print('=== phase 1: training emotion model head ===')
history_emo_p1 = emo_model.fit(
    emo_train,
    validation_data=emo_val,
    epochs=10,
    class_weight=emo_class_weight,
    callbacks=callbacks_emo,
    verbose=1
)

In [ ]:
# phase 2 for emotion - fine-tune
print('=== phase 2: fine-tuning emotion model ===')
emo_base.trainable = True
for layer in emo_base.layers[:-30]:
    layer.trainable = False

emo_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                  loss='categorical_crossentropy', metrics=['accuracy'])

history_emo_p2 = emo_model.fit(
    emo_train,
    validation_data=emo_val,
    epochs=10,
    class_weight=emo_class_weight,
    callbacks=callbacks_emo,
    verbose=1
)

history_emo = merge_hist(history_emo_p1, history_emo_p2)
print('emotion model training finished')

## Learning Curves - checking for overfitting or underfitting

if training accuracy is much higher than validation accuracy -> overfitting
if both are low -> underfitting

In [ ]:
def plot_learning_curves(history_dict, model_name, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Learning Curves - {model_name}', fontsize=13, fontweight='bold')

    epochs = range(1, len(history_dict['loss']) + 1)

    # loss
    axes[0].plot(epochs, history_dict['loss'], 'b-o', label='train loss', markersize=3)
    axes[0].plot(epochs, history_dict['val_loss'], 'r-o', label='val loss', markersize=3)
    axes[0].set_title('Loss')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # accuracy
    axes[1].plot(epochs, history_dict['accuracy'], 'b-o', label='train accuracy', markersize=3)
    axes[1].plot(epochs, history_dict['val_accuracy'], 'r-o', label='val accuracy', markersize=3)
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    print(f'saved to {save_path}')

plot_learning_curves(history_mask, 'Mask Model (MobileNetV2)', 'outputs/learning_curve_mask.png')
plot_learning_curves(history_emo,  'Emotion Model (MobileNetV2)', 'outputs/learning_curve_emotion.png')

## Step 8 - Evaluation

i will evaluate both models using:
- accuracy
- precision
- recall
- f1 score
- confusion matrix
- ROC-AUC (for mask model since it's binary)

In [ ]:
# helper to get predictions from a generator
def predict_binary(model, generator):
    generator.reset()
    y_true = []
    y_proba = []
    for i in range(len(generator)):
        X, y = generator[i]
        preds = model.predict(X, verbose=0).flatten()
        y_true.extend(y.tolist())
        y_proba.extend(preds.tolist())
    y_true  = np.array(y_true)
    y_proba = np.array(y_proba)
    y_pred  = (y_proba >= 0.5).astype(int)
    return y_true, y_proba, y_pred

def predict_multiclass(model, generator):
    generator.reset()
    y_true = []
    y_pred = []
    for i in range(len(generator)):
        X, y = generator[i]
        preds = model.predict(X, verbose=0)
        y_true.extend(np.argmax(y, axis=1).tolist())
        y_pred.extend(np.argmax(preds, axis=1).tolist())
    return np.array(y_true), np.array(y_pred)

# get predictions
print('getting mask model predictions...')
mask_y_true, mask_y_proba, mask_y_pred = predict_binary(mask_model, mask_val)

print('getting emotion model predictions...')
emo_y_true, emo_y_pred = predict_multiclass(emo_model, emo_val)

print('done')

In [ ]:
# print metrics for mask model
mask_classes = ['without_mask', 'with_mask']

print('=== mask model evaluation ===')
print(f'accuracy : {accuracy_score(mask_y_true, mask_y_pred):.4f}')
print(f'precision: {precision_score(mask_y_true, mask_y_pred):.4f}')
print(f'recall   : {recall_score(mask_y_true, mask_y_pred):.4f}')
print(f'f1 score : {f1_score(mask_y_true, mask_y_pred):.4f}')
print(f'roc auc  : {roc_auc_score(mask_y_true, mask_y_proba):.4f}')
print()
print('detailed per-class report:')
print(classification_report(mask_y_true, mask_y_pred, target_names=mask_classes))

In [ ]:
# print metrics for emotion model
print('=== emotion model evaluation ===')
print(f'accuracy : {accuracy_score(emo_y_true, emo_y_pred):.4f}')
print(f'precision: {precision_score(emo_y_true, emo_y_pred, average="weighted"):.4f}')
print(f'recall   : {recall_score(emo_y_true, emo_y_pred, average="weighted"):.4f}')
print(f'f1 score : {f1_score(emo_y_true, emo_y_pred, average="weighted"):.4f}')
print()
print('detailed per-class report:')
print(classification_report(emo_y_true, emo_y_pred, target_names=EMOTION_LABELS))

In [ ]:
# confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')

# mask confusion matrix
cm_mask = confusion_matrix(mask_y_true, mask_y_pred)
sns.heatmap(cm_mask, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=mask_classes, yticklabels=mask_classes)
axes[0].set_title('Mask Detection')
axes[0].set_xlabel('predicted')
axes[0].set_ylabel('actual')

# emotion confusion matrix
cm_emo = confusion_matrix(emo_y_true, emo_y_pred)
sns.heatmap(cm_emo, annot=True, fmt='d', cmap='Purples', ax=axes[1],
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS)
axes[1].set_title('Emotion Detection')
axes[1].set_xlabel('predicted')
axes[1].set_ylabel('actual')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('outputs/confusion_matrices.png')
plt.show()
print('saved outputs/confusion_matrices.png')

In [ ]:
# ROC curve for mask model (since it's binary classification)
auc_score = roc_auc_score(mask_y_true, mask_y_proba)
fpr, tpr, _ = roc_curve(mask_y_true, mask_y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='random classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
ax.set_xlabel('false positive rate')
ax.set_ylabel('true positive rate')
ax.set_title('ROC Curve - Mask Detection', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/roc_curve_mask.png')
plt.show()
print('saved outputs/roc_curve_mask.png')

## Baseline vs Transfer Learning Comparison

i will compare the simple CNN with the MobileNetV2 model to show that transfer learning is better.

In [ ]:
# get baseline predictions
bl_mask_true, bl_mask_proba, bl_mask_pred = predict_binary(baseline_mask, mask_val)
bl_emo_true, bl_emo_pred                  = predict_multiclass(baseline_emo, emo_val)

# build comparison table
comparison = {
    'model': [
        'Baseline CNN - Mask',
        'MobileNetV2  - Mask',
        'Baseline CNN - Emotion',
        'MobileNetV2  - Emotion'
    ],
    'accuracy': [
        round(accuracy_score(bl_mask_true, bl_mask_pred), 4),
        round(accuracy_score(mask_y_true, mask_y_pred), 4),
        round(accuracy_score(bl_emo_true, bl_emo_pred), 4),
        round(accuracy_score(emo_y_true, emo_y_pred), 4)
    ],
    'f1 score': [
        round(f1_score(bl_mask_true, bl_mask_pred), 4),
        round(f1_score(mask_y_true, mask_y_pred), 4),
        round(f1_score(bl_emo_true, bl_emo_pred, average='weighted'), 4),
        round(f1_score(emo_y_true, emo_y_pred, average='weighted'), 4)
    ],
    'roc auc': [
        round(roc_auc_score(bl_mask_true, bl_mask_proba), 4),
        round(roc_auc_score(mask_y_true, mask_y_proba), 4),
        'n/a', 'n/a'
    ]
}

comp_df = pd.DataFrame(comparison)
print('=== BASELINE vs TRANSFER LEARNING ===')
print(comp_df.to_string(index=False))

In [ ]:
# bar chart comparison
labels  = ['Mask Detection', 'Emotion Detection']
bl_accs = [accuracy_score(bl_mask_true, bl_mask_pred),
            accuracy_score(bl_emo_true, bl_emo_pred)]
tl_accs = [accuracy_score(mask_y_true, mask_y_pred),
            accuracy_score(emo_y_true, emo_y_pred)]

x = np.arange(len(labels))
w = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, bl_accs, w, label='Baseline CNN', color='salmon', edgecolor='black')
bars2 = ax.bar(x + w/2, tl_accs, w, label='MobileNetV2', color='steelblue', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel('accuracy')
ax.set_title('Baseline CNN vs MobileNetV2 Transfer Learning', fontsize=13, fontweight='bold')
ax.legend()

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/baseline_vs_transfer.png')
plt.show()
print('saved outputs/baseline_vs_transfer.png')

In [ ]:
# save the final models
mask_model.save('models/mask_model_final.h5')
emo_model.save('models/emotion_model_final.h5')

print('models saved:')
print('  models/mask_model_final.h5')
print('  models/emotion_model_final.h5')
print()
print('download these files and put them in the models/ folder locally')
print('then run app.py for the live webcam demo')

## Summary

what i did in this notebook:

| step | what i did |
|------|------------|
| data prep | cropped face ROIs from mask dataset, set up ImageDataGenerators with augmentation |
| baseline  | trained simple 3-block CNN for mask and emotion (for comparison) |
| transfer  | used MobileNetV2 pretrained on ImageNet, added custom head |
| overfitting | added dropout, L2 regularization, early stopping, ReduceLR |
| fine-tuning | unfroze last 30 layers and retrained with small learning rate |
| evaluation | accuracy, precision, recall, F1, confusion matrix, ROC-AUC |
| comparison | baseline CNN vs MobileNetV2 side by side |

the models are saved in the `models/` folder.
download them and put in your local repo, then run `app.py`.